# Statim: train the multilingual decision model on an A100 (one click)

**Run: Runtime → Run all (Ctrl+F9).** The A100 is preselected. The secret `HF_TOKEN` (key icon) must allow notebook access.

Recipe: the v9 recipe (released 0.7.0), unchanged except for more data per epoch. The A100 makes that affordable;
v9 saw only 24,000 of the 725,000 mixture items per epoch.
- Start: laya-multilingual-big1 (0.4.0), whose encoder is jhu-clsp/mmBERT-base (307M).
- Data: mixture-v8, the v9 training mixture (licence-audited, no synthetic data), plus Banking77, MASSIVE (51 languages) and typed decisions.
- Distillation from the start model keeps what it knew (learning without forgetting).
- EMA 0.999, warmup 6 %, early stop after 3 epochs without a new best dev mean, bf16.

Inputs come from the **private** repo `Beko2210/statim-train-staging`, and the trained model goes back there under `runs/`.
Nothing is published. The gate (pop-os) decides whether the model ships.


In [ ]:
# ===== SETUP (~5 min) =====
import os, subprocess, sys, time, json, threading
RUN = "v11-a100"
STAGING = "Beko2210/statim-train-staging"          # private
BUDGET = "banking77=12000,massive=24000,mixture=96000,typed=6000,distill=18000"

def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise SystemExit("FAILED: " + cmd)
    return r.stdout

print("GPU:", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").strip())
from google.colab import userdata
os.environ["HF_TOKEN"] = HF_TOKEN = userdata.get("HF_TOKEN")
sh("pip -q install laya==0.3.20 datasets huggingface_hub safetensors")
os.chdir("/content")
sh("rm -rf /content/statim && git clone -q --depth 1 -b train/colab-a100 https://github.com/BEKO2210/statim /content/statim")
os.chdir("/content/statim")
print(sh("git log --oneline -1"))
from huggingface_hub import HfApi, snapshot_download
api = HfApi(token=HF_TOKEN)
snapshot_download(STAGING, token=HF_TOKEN, local_dir="/content/stage",
                  allow_patterns=["base/laya-multilingual-big1/*", "base/laya-multilingual-big1/**", "data/mixture-v8.jsonl.gz"])
print("RAM GB:", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9))
print("SETUP OK")


In [ ]:
# ===== TRAIN (~2.5-3.5 h; live log; the finished model is uploaded to the private repo) =====
OUT = f"/content/{RUN}"
LOG = f"/content/{RUN}.log"
cmd = [sys.executable, "-u", "tools/finetune/train_multitask.py", "/content/stage/base/laya-multilingual-big1", OUT,
       "--clean", "--mixture", "/content/stage/data/mixture-v8.jsonl.gz", "--mixture-dev", "6000",
       "--massive-per-lang", "11000", "--max-len", "1024", "--epochs", "12", "--patience", "3",
       "--distill", "24000", "--warmup", "0.06", "--ema", "0.999", "--budget", BUDGET,
       "--max-tokens", "16384", "--accum", "1"]
print(" ".join(cmd), flush=True)

stop = threading.Event()
def push_log():  # the log reaches the private repo every 10 minutes, so progress survives a disconnect
    while not stop.wait(600):
        try:
            api.upload_file(path_or_fileobj=LOG, path_in_repo=f"runs/{RUN}/train.log", repo_id=STAGING,
                            commit_message=f"{RUN}: log")
        except Exception as exc:
            print("log upload failed (will retry):", type(exc).__name__, flush=True)
threading.Thread(target=push_log, daemon=True).start()

t0 = time.time()
with open(LOG, "w") as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        log.write(line); log.flush()
        if not line.startswith("  epoch") or " step " not in line or "step 0/" in line or int(line.split("step ")[1].split("/")[0]) % 2000 == 0:
            print(line, end="", flush=True)
    code = proc.wait()
stop.set()
print(f"exit {code} after {(time.time() - t0) / 3600:.2f} h", flush=True)
api.upload_file(path_or_fileobj=LOG, path_in_repo=f"runs/{RUN}/train.log", repo_id=STAGING, commit_message=f"{RUN}: final log")
if code:
    raise SystemExit("training failed (see the log above)")
api.upload_folder(folder_path=OUT, path_in_repo=f"runs/{RUN}/model", repo_id=STAGING, commit_message=f"{RUN}: trained model")
print("uploaded", f"{STAGING}/runs/{RUN}/model (private). Next: the gate on pop-os.")
